In [ ]:
# ================================
# 3. Association Rule Mining
# FP-growth on mobile_price.csv
# ================================

import pandas as pd
import numpy as np

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules


# ================================
# Load dataset
# ================================

df = pd.read_csv("mobile_price.csv")

# Filter only samples where price_range = 1
df_price_1 = df[df["price_range"] == 1].copy()

# Select four features
features = ["ram", "int_memory", "px_width", "battery_power"]
df_selected = df_price_1[features].copy()

print("Filtered dataset shape:", df_selected.shape)
display(df_selected.head())

In [ ]:
# ================================
# Categorize features into low / medium / high
# using 3:4:3 ratio
# ================================

def categorize_343(series):
    min_val = series.min()
    max_val = series.max()
    value_range = max_val - min_val

    low_threshold = min_val + 0.3 * value_range
    high_threshold = min_val + 0.7 * value_range

    def categorize_value(x):
        if x <= low_threshold:
            return "low"
        elif x <= high_threshold:
            return "medium"
        else:
            return "high"

    return series.apply(categorize_value)


df_categorized = pd.DataFrame()

for feature in features:
    df_categorized[feature] = categorize_343(df_selected[feature])

print("Categorized data:")
display(df_categorized.head())

In [ ]:
# ================================
# Convert each row into transaction format
# Example:
# ram_high, int_memory_low, px_width_low, battery_power_low
# ================================

transactions = []

for _, row in df_categorized.iterrows():
    transaction = []
    for feature in features:
        item = feature + "_" + row[feature]
        transaction.append(item)
    transactions.append(transaction)

print("First 5 transactions:")
for t in transactions[:5]:
    print(t)

In [ ]:
# ================================
# One-hot encode transactions
# ================================

te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)

df_transactions = pd.DataFrame(te_array, columns=te.columns_)

print("Transaction table:")
display(df_transactions.head())

In [ ]:
# ================================
# Apply FP-growth algorithm
# ================================

# You can adjust min_support if too few or too many itemsets are generated
min_support = 0.3

frequent_itemsets = fpgrowth(
    df_transactions,
    min_support=min_support,
    use_colnames=True
)

frequent_itemsets = frequent_itemsets.sort_values(
    by="support",
    ascending=False
).reset_index(drop=True)

print("Frequent itemsets:")
display(frequent_itemsets)

In [ ]:
# ================================
# Generate association rules
# ================================
# Filter rules by support, confidence, and lift
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.4
)
rules_filtered = rules[
    (rules["support"] >= 0.3) &
    (rules["confidence"] >= 0.4) &
    (rules["lift"] >= 0.8)
].copy()
# Sort rules
rules_filtered = rules_filtered.sort_values(
    by=["support", "confidence", "lift"],
    ascending=False
).reset_index(drop=True)

# Select useful columns
rules_summary = rules_filtered[
    [
        "antecedents",
        "consequents",
        "antecedent support",
        "consequent support",
        "support",
        "confidence",
        "lift"
    ]
].copy()
# Make rules easier to read
rules_summary["antecedents"] = rules_summary["antecedents"].apply(
    lambda x: ", ".join(sorted(list(x)))
)

rules_summary["consequents"] = rules_summary["consequents"].apply(
    lambda x: ", ".join(sorted(list(x)))
)

print("===== 3(b) Association rules =====")
print("Rules with support >= 0.3, confidence >= 0.4, and lift >= 0.8")
display(rules_summary)

In [ ]:
# ================================
# 4. PCA and K-Means
# ================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics.cluster import adjusted_rand_score

In [ ]:
# ================================
# 4(a) Load data and standardize features
# ================================

df = pd.read_csv('mobile_price.csv')

# Split features and labels
X = df
y = df["price_range"]

# Z-score standardization
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Original feature shape:", X.shape)
print("Standardized feature shape:", X_scaled.shape)

# Optional: check mean and std after standardization
print("Mean after standardization:", np.round(X_scaled.mean(axis=0), 4))
print("Std after standardization:", np.round(X_scaled.std(axis=0), 4))

In [ ]:
# ================================
# 4(b) PCA projection to 2 dimensions
# ================================

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(
    X_pca,
    columns=["PC1", "PC2"]
)

pca_df["price_range"] = y.values

print("Explained variance ratio:")
print("PC1:", pca.explained_variance_ratio_[0])
print("PC2:", pca.explained_variance_ratio_[1])
print("Total:", pca.explained_variance_ratio_.sum())

display(pca_df.head())

In [ ]:
# ================================
# Visualize PCA result with true class labels
# ================================

plt.figure(figsize=(8, 6))

for label in sorted(pca_df["price_range"].unique()):
    subset = pca_df[pca_df["price_range"] == label]
    plt.scatter(
        subset["PC1"],
        subset["PC2"],
        label=f"Class {label}",
        alpha=0.7
    )

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("PCA Projection of Mobile Price Data")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ================================
# 4(c) K-Means clustering using all standardized features
# ================================

kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans.fit_predict(X_scaled)

# Evaluate clustering performance using Adjusted Rand Score
ari_score = adjusted_rand_score(y, cluster_labels)

print("===== K-Means Clustering Performance =====")
print("Adjusted Rand Score:", ari_score)

In [ ]:
# ================================
# Visualize K-Means clustering result on PCA space
# ================================

cluster_df = pd.DataFrame(
    X_pca,
    columns=["PC1", "PC2"]
)

cluster_df["cluster"] = cluster_labels
cluster_df["true_label"] = y.values

plt.figure(figsize=(8, 6))

for cluster in sorted(cluster_df["cluster"].unique()):
    subset = cluster_df[cluster_df["cluster"] == cluster]
    plt.scatter(
        subset["PC1"],
        subset["PC2"],
        label=f"Cluster {cluster}",
        alpha=0.7
    )

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("K-Means Clustering Results Visualized by PCA")
plt.legend()
plt.grid(True)
plt.show()